## Preprocessing and filtering data


Set path to include RStudio packages

In [1]:
.libPaths(c("/net/home/swolf/R/x86_64-pc-linux-gnu-library/4.4", .libPaths()))

In [2]:
library(data.table)
library(tidyverse)
library(CoordinateCleaner)
library(sf)
library(parallel)
library(dggridR)
library(terra) #for intersection of points
library(readxl) #to read GloNAF files


setwd("~/Func_Neo_Phytes")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 

## Import and initial filtering of GBIF data

In [3]:
library(data.table)
dtn = 10
setDTthreads(threads = dtn) #defines how many threads data.table is allowed to use

In [4]:
# GBIF full dataset
file_csv <- "/net/data/GBIF/2024_04_26/0001058-240425142415019.csv"

# alternative file citizen scince only
file_csv <- "/net/data/GBIF/2025_06_15_cit_science_only/0047329-250525065834625.csv"

test <- fread(file_csv, nrow = 1000, na.strings = "", quote = "")
head(test)

gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,⋯,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
<int64>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<lgl>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<chr>,<lgl>,<chr>
3352641301,0a013f89-5381-4578-9d82-5f28fd5f1ef6,7213446,Plantae,Tracheophyta,Magnoliopsida,Asterales,Asteraceae,Matricaria,Matricaria discoidea,⋯,"Conservation Data Management Unit, RSPB",NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,"Christie, M. Michael",NA,NA,2025-06-11T15:06:08.718Z,NA,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED
4951294464,0a013f89-5381-4578-9d82-5f28fd5f1ef6,36680109,Plantae,Tracheophyta,Liliopsida,Poales,Poaceae,Deschampsia,Deschampsia cespitosa,⋯,"Common, James",NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,"Common, J. James",NA,NA,2025-06-11T15:06:08.739Z,NA,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED
4866764380,0a013f89-5381-4578-9d82-5f28fd5f1ef6,34977000,Plantae,Tracheophyta,Magnoliopsida,Fagales,Fagaceae,Quercus,Quercus robur,⋯,"Common, James",NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,"Common, J. James",NA,NA,2025-06-11T15:07:00.854Z,NA,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED
3098399480,0a013f89-5381-4578-9d82-5f28fd5f1ef6,12254837,Plantae,Tracheophyta,Magnoliopsida,Saxifragales,Haloragaceae,Myriophyllum,Myriophyllum verticillatum,⋯,Norfolk Flora Group,NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,Norfolk Flora Group,NA,NA,2025-06-11T15:07:00.854Z,NA,INDIVIDUAL_COUNT_INVALID;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED
3352641302,0a013f89-5381-4578-9d82-5f28fd5f1ef6,18230781,Plantae,Tracheophyta,Polypodiopsida,Polypodiales,Aspleniaceae,Asplenium,Asplenium trichomanes,⋯,"Shuttleworth, Ali",NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,"Shuttleworth, A. Ali",NA,NA,2025-06-11T15:06:08.769Z,NA,INDIVIDUAL_COUNT_INVALID;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED
5158856302,0a013f89-5381-4578-9d82-5f28fd5f1ef6,40476637,Plantae,Tracheophyta,Magnoliopsida,Ericales,Primulaceae,Primula,Primula veris,⋯,"Pescott, Oli",NA,CC_BY_4_0,Botanical Society of Britain and Ireland and Biological Records Centre,"Pescott, O. Oli",NA,NA,2025-06-11T15:06:08.782Z,NA,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_MATCH_TAXON_ID_IGNORED;TAXON_MATCH_TAXON_CONCEPT_ID_IGNORED


Raw data: 285,132,563 observations
Raw data cit science only: 75261671

In [ ]:
columns  <- c("gbifID", "class", "speciesKey", "species","decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters", "establishmentMeans")

gbif_dt <- fread(file_csv, select = columns, na.strings = "", quote = "") %>% #, nrow = 100000
  filter(!is.na(speciesKey)) %>% # return taxons at the level of species and below, i.e. subspecies, varieties, and forms
  filter(!class %in% c("Polypodiopsida","Lycopodiopsida", NA)) %>% # filtering out ferns and mosses as we are interested only in seed plants
  filter(!coordinateUncertaintyInMeters %in% c(301, 3036, 999, 9999)) %>% # exclude errors that produced by geocoding software and do not represent real uncertainty values
  filter(coordinateUncertaintyInMeters <= 10000 | is.na(coordinateUncertaintyInMeters)) %>% # exclude records with too high uncertainty values
  #distinct(species, establishmentMeans, round(decimalLatitude, 4), round(decimalLongitude, 4), .keep_all = T) %>% # SW: I removed this because we want the abundances fo our analysis select only unique observations by species and establishment means as we are interested in pres-abs, but not abundances. The idea of trimming coordinates to 4 digits https://peerj.com/articles/9916.pdf
  select(gbifID, speciesKey, class, species, decimalLatitude, decimalLongitude, establishmentMeans)

head(gbif_dt)

In [ ]:
# Number of observations after first filtering step
dim(gbif_dt)[1]

In [ ]:
# Number of species 
length(unique(gbif_dt[["speciesKey"]]))

In [ ]:
# Percentage of observations kept
#dim(gbif_dt)[1]/285132563 # all GBIF data
dim(gbif_dt)[1]/75261671 # citizen science only

In [ ]:
# Save filtering step as file

fwrite(gbif_dt, "Results/Data/01_prep_gbif-import-init_cs.csv", row.names = F)
rm(list = ls())
gc()

This resolves into 261,756,861 records (91% from the initial amount) and 261,251 species by species names for whole GBIF. With citizen scince only, 92% kept, with 69703083 records and 114322 species.


## Spatial Cleaning

We know from the previous step that the total number of records is 261,756,861. We now split them into separate chunks \~100,000 records per chunk, so that we can read and process the chunks in parallel


In [ ]:
#gbif_dt <- fread("Results/Data/01_prep_gbif-import-init.csv")
gbif_dt <- fread("Results/Data/01_prep_gbif-import-init_cs.csv")  # for cs only

head(gbif_dt)

In [ ]:
obs_num <- dim(gbif_dt)[1]

#chunks <- split(1:obs_num, sort(rep_len(1:1500, length(1:obs_num))))
chunks <- split(1:obs_num, sort(rep_len(1:100, length(1:obs_num)))) # for cs only

In [ ]:
cc_fun <- function(i){
  
  filename <- paste0("/net/scratch/swolf/GBIF/gbif_cclean/01_prep_gbif-cclean-chunk", i ,".csv")
  
  if(!file.exists(filename)){
    gbif_chunk_dt <- fread("Results/Data/01_prep_gbif-import-init_cs.csv", na.strings = "",
                           nrow = length(chunks[[i]]), skip = chunks[[i]][1]-1,
                           colClasses = c("integer", "integer", rep("character", 2),
                                          rep("numeric", 2), "character"))
    colnames(gbif_chunk_dt) <- c("gbifID", "speciesKey", "class", "species",
                                 "decimalLatitude", "decimalLongitude",
                                 "establishmentMeans")
    
    gbif_clean_dt <- cc_cap(gbif_chunk_dt, buffer = 5000) %>%
      cc_cen(buffer = 5000) %>% #when there is no centroid in the area, it throws an error https://github.com/ropensci/CoordinateCleaner/issues/89, should be corrected when 3.0 matures
      cc_aohi(taxa = "Plantae", buffer = 5000) %>%
      cc_inst(buffer = 1000) %>%
      cc_gbif(buffer = 1000) %>%
      cc_zero() %>%
      cc_equ() %>%
      cc_sea(ref = terra::unwrap(buffland), scale = 10)
    
    fwrite(gbif_clean_dt, filename, row.names = F, quote = T)
    rm(gbif_chunk_dt, gbif_clean_dt)
    gc(verbose = FALSE)
  }
  
}


cl <- makeForkCluster(10)
begin <- Sys.time()
parLapplyLB(cl, seq_along(chunks), cc_fun)
end <- Sys.time()
print(end-begin)
stopCluster(cl)

#read and process rows across the index to merge them into a single data table
#when reading make sure using column classes
chunk_files <- list.files("/net/scratch/swolf/GBIF/gbif_cclean/", full.names = T)
#to check if any chunk was unprocessed (because of absent centorid)
nmbr <- as.numeric(lapply(str_extract_all(chunk_files, "\\d+"), "[[", 2))

if (length(setdiff(seq_along(chunks), nmbr)) == 0) {
  gbif_cclean_dt <- do.call(rbind, lapply(chunk_files, fread, na.strings = "",
                                          colClasses = c("integer", rep("character", 2),
                                                         rep("numeric", 2), "character")))
} else {
  print("There are", length(setdiff(seq_along(chunks), nmbr)), "files missing")
}

dim(gbif_cclean_dt)
length(unique(gbif_cclean_dt[["speciesKey"]]))


fwrite(gbif_cclean_dt, "Results/Data/01_prep_gbif-coord-clean.csv", row.names = F, quote = T)

rm(list = ls())
gc()

In [ ]:
# Same code for citizen science only:

cc_fun <- function(i){
  
  filename <- paste0("/net/scratch/swolf/GBIF/gbif_cclean/01_prep_gbif-cclean-chunk", i ,".csv")
  
  if(!file.exists(filename)){
    gbif_chunk_dt <- fread("Results/Data/01_prep_gbif-import-init_cs.csv", na.strings = "",
                           nrow = length(chunks[[i]]), skip = chunks[[i]][1]-1,
                           colClasses = c("integer", "integer", rep("character", 2),
                                          rep("numeric", 2), "character"))
    colnames(gbif_chunk_dt) <- c("gbifID", "speciesKey", "class", "species",
                                 "decimalLatitude", "decimalLongitude",
                                 "establishmentMeans")
    
    gbif_clean_dt <- cc_cap(gbif_chunk_dt, buffer = 5000) %>%
      cc_cen(buffer = 1000) %>% #when there is no centroid in the area, it throws an error https://github.com/ropensci/CoordinateCleaner/issues/89, should be corrected when 3.0 matures
      #cc_aohi(taxa = "Plantae", buffer = 5000) %>%
      #cc_inst(buffer = 1000) %>%
      cc_gbif(buffer = 1000) %>%
      cc_zero() %>%
      cc_equ() %>%
      cc_sea(ref = terra::unwrap(buffland), scale = 10)
    
    fwrite(gbif_clean_dt, filename, row.names = F, quote = T)
    rm(gbif_chunk_dt, gbif_clean_dt)
    gc(verbose = FALSE)
  }
  
}


cl <- makeForkCluster(10)
begin <- Sys.time()
parLapplyLB(cl, seq_along(chunks), cc_fun)
end <- Sys.time()
print(end-begin)
stopCluster(cl)


In [ ]:
# Merge citizen science only files:

#read and process rows across the index to merge them into a single data table
#when reading make sure using column classes
chunk_files <- list.files("/net/scratch/swolf/GBIF/gbif_cclean/", full.names = T)
#to check if any chunk was unprocessed (because of absent centorid)
nmbr <- as.numeric(lapply(str_extract_all(chunk_files, "\\d+"), "[[", 2))

if (length(setdiff(seq_along(chunks), nmbr)) == 0) {
  gbif_cclean_dt <- do.call(rbind, lapply(chunk_files, fread, na.strings = "",
                                          colClasses = c("integer", "integer", qrep("character", 2),
                                                         rep("numeric", 2), "character")))
} else {
  print("There are", length(setdiff(seq_along(chunks), nmbr)), "files missing")
}

dim(gbif_cclean_dt)
length(unique(gbif_cclean_dt[["speciesKey"]]))


#fwrite(gbif_cclean_dt, "Results/Data/01_prep_gbif-coord-clean.csv", row.names = F, quote = T)
fwrite(gbif_cclean_dt, "Results/Data/01_prep_gbif-coord-clean_sc.csv", row.names = F, quote = T)

rm(list = ls())
gc()

In [4]:
gbif_cclean_dt <- fread("Results/Data/01_prep_gbif-coord-clean_sc.csv")
head(gbif_cclean_dt)

gbifID,speciesKey,class,species,decimalLatitude,decimalLongitude,establishmentMeans
<int64>,<int>,<chr>,<chr>,<dbl>,<dbl>,<chr>
3352641301,3103791,Magnoliopsida,Matricaria discoidea,56.14011,-3.369429,
4951294464,8059811,Liliopsida,Deschampsia cespitosa,54.97456,-1.634406,
4866764380,2878688,Magnoliopsida,Quercus robur,54.99222,-1.540470,
3098399480,5361779,Magnoliopsida,Myriophyllum verticillatum,52.75213,1.511204,
5158856302,8377689,Magnoliopsida,Primula veris,51.60241,-1.122803,
3098399483,5371825,Magnoliopsida,Aethusa cynapium,51.10395,-1.993522,


In [ ]:
dim(gbif_cclean_dt)
length(unique(gbif_cclean_dt[["speciesKey"]]))
dim(gbif_cclean_dt)[1]/69703083
dim(gbif_cclean_dt)[1]/75261671

In [ ]:
# Stats of all GBIF
dim(gbif_cclean_dt) # 241975710

length(unique(gbif_cclean_dt[["species"]])) # 257299

In [5]:
241975710/261756861 #92.4% of observations

257299/260398 #98.8% species

[1] 0.9244293

[1] 0.988099